In [1]:
import pandas as pd
import numpy as np
import torch
import os
import sys
from tqdm import tqdm, trange

sys.path.append("../../")
import biked_commons
from biked_commons.design_evaluation.design_evaluation import *
from biked_commons.resource_utils import split_datasets_path
from biked_commons.conditioning import conditioning
from biked_commons.design_evaluation.scoring import *

c:\Users\Lyler\mambaforge\envs\torch\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\Lyler\mambaforge\envs\torch\Lib\site-packages\sklearn\base.py:380: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.6.1 when using version 1.6.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\Lyler\Documents\biked-commons\src\biked_commons\design_evaluation\../..\biked_commons\prediction\usability_predictors.py:37: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data whic

In [2]:
data = pd.read_csv(split_datasets_path("bike_bench.csv"), index_col=0)

#sample 100
data = data.sample(100, random_state=0)
data_tens = torch.tensor(data.values, dtype=torch.float32)

In [3]:
evaluator, requirement_names, requirement_types = construct_tensor_evaluator(StandardEvaluations, data.columns)
isobjective = torch.tensor(requirement_types) == 1


In [4]:
num_data = data.shape[0]
rider_condition = conditioning.sample_riders(num_data, split="test")
use_case_condition = conditioning.sample_use_case(num_data, split="test")
text_condition = conditioning.sample_text(num_data, split="test")
image_embeddings = conditioning.sample_image_embedding(num_data, split="test")
condition = {"Rider": rider_condition, "Use Case": use_case_condition, "Embedding": image_embeddings}
# condition = {"Rider": rider_condition, "Use Case": use_case_condition, "Text": text_condition}

In [5]:
eval_scores = evaluator(data_tens, condition)

c:\Users\Lyler\mambaforge\envs\torch\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


In [6]:
#check gradient of eval scores wrt data_tens
data_tens.requires_grad = True
eval_scores = evaluator(data_tens, condition)
eval_scores_sum = eval_scores.sum()
eval_scores_sum.backward()
print(data_tens.grad.shape)
print(data_tens.grad[0])



c:\Users\Lyler\mambaforge\envs\torch\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


torch.Size([100, 97])
tensor([-2.6405e+00,  3.1375e+00,  1.7829e+00,  8.2237e+00, -3.8171e+00,
         5.8137e-01, -5.7944e-01, -5.5592e+00, -2.5429e+00, -1.0149e+00,
        -2.3912e-02,  3.5576e-03,  8.8400e-02,  2.6578e-02,  1.9905e-02,
         4.4998e-02, -1.5685e-04,  9.9779e-01, -3.1325e-03,  3.8234e+00,
         5.2103e-04,  8.1453e-03,  1.8365e-02, -1.6776e-01,  4.4507e-02,
        -2.4693e-03,  8.3375e-03,  1.6553e-02,  2.2097e-02,  6.5353e-02,
         2.0612e-01,  1.3668e-01, -7.8637e-03,  0.0000e+00,  5.0000e-01,
         0.0000e+00,  5.0000e-01,  0.0000e+00,  0.0000e+00,  0.0000e+00,
        -2.4339e-03, -1.7970e-03,  0.0000e+00, -3.3185e-03,  0.0000e+00,
        -1.0000e+00, -1.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
         0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
         0.0000e+00, -1.4067e-01,  0.0000e+00, -1.0000e+00, -1.0834e+00,
         0.0000e+00,  0.0000e+00,  0.0000e+00,  4.7246e-01, -3.9902e-01,
         0.0000e+00,  0.0000e

In [7]:
isobjective = torch.tensor(requirement_types) == 1
objective_scores = eval_scores[:, isobjective].detach().numpy()
# constraint_scores = eval_scores[:, ~isobjective].detach().numpy()

In [8]:
main_scorer = construct_scorer(MainScores, StandardEvaluations, data.columns)
detailed_scorer = construct_scorer(DetailedScores, StandardEvaluations, data.columns)

In [9]:
main_scorer(data_tens.detach(), condition)

c:\Users\Lyler\mambaforge\envs\torch\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


Hypervolume                     0.000000
Constraint Satisfaction Rate    0.874000
Maximum Mean Discrepancy        0.003185
dtype: float64

In [10]:
detailed_scorer(data_tens.detach(), condition)

c:\Users\Lyler\mambaforge\envs\torch\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


Min Objective Score: Usability Score - 0 to 1                                                                 0.791411
Min Objective Score: Drag Force                                                                              27.965340
Min Objective Score: Knee Angle Error                                                                       186.338130
Min Objective Score: Hip Angle Error                                                                        809.888700
Min Objective Score: Arm Angle Error                                                                        848.619570
Min Objective Score: Mass                                                                                    22.190498
Min Objective Score: Planar Compliance                                                                      180.326250
Min Objective Score: Transverse Compliance                                                                  265.009770
Min Objective Score: Eccentric Compliance       